# 第71章 交互直方图（px.histogram）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 8 / 18 步：交互观察分布与矩阵**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互面积图（px.area）  →  **本章任务：** 交互直方图（px.histogram）  →  **下一步：** 交互箱线图（px.box）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

面对订单金额、商品价格这样一整列连续数字，光靠表格很难看出数据集中在哪个区间、分布是否偏斜。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互直方图（px.histogram）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互直方图（px.histogram）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互直方图（px.histogram）」并读出其中的结论。


## 适用场景

**背景引入**：面对订单金额、商品价格这样一整列连续数字，光靠表格很难看出数据集中在哪个区间、分布是否偏斜。直方图把这些数值按区间分箱，用柱形高低把“大部分数据落在哪、两头有多稀疏”一眼摊开，是探索分布的第一步。Plotly 的交互版本还能把鼠标放上去直接读出每个区间的数量，并一键加上分组、密度和边缘图，特别适合做初步的数据体检。（可以把它想成把一整列数字“称重分堆”：先画好一排筐（分箱），每筐装一个数值区间的数，柱高就是这筐塞了多少个；筐分得粗或细，看到的分布轮廓就不一样，所以改 nbins 就是在换筐的大小。）

探索连续变量的频数、密度和分组形状。


## 数据结构

一列连续数值和可选分类列。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 nbins 从 20 改为 12 或 30，观察分箱数量对分布形状的影响
2. 修改 barmode="overlay" 为 "stack"，对比叠加与堆积对分组分布比较的作用
3. 添加 marginal="violin" 参数，观察边缘图对分布形状的补充展示


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.histogram()`、`fig.update_layout()`、`fig.show()` | 探索连续变量的频数、密度和分组形状。 | 分组时分箱边界不一致 |
| 进阶变体 | `px.histogram()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 图例筛选后忘记样本量改变 |
| 关键参数 | `nbins` | 分箱数 | 分组时分箱边界不一致 |
| 关键参数 | `histnorm` | 归一化 | 图例筛选后忘记样本量改变 |
| 关键参数 | `barmode` | overlay/stack | 过度依赖默认分箱 |
| 关键参数 | `marginal` | 边缘图 | 分组时分箱边界不一致 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-71 -->
### 数学推导｜直方图的频数与密度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜用箱边界划分数轴。** 第 $j$ 个箱为 $[b_j,b_{j+1})$，箱宽 $h_j=b_{j+1}-b_j$。

**第 2 步｜数落入箱中的样本。** $n_j=\sum_i\mathbf{1}(b_j\le x_i<b_{j+1})$，相对频率为 $n_j/n$。

**第 3 步｜让柱形面积代表概率。** 柱高应满足“高 × 宽 = 相对频率”，所以

$$
\hat f_jh_j=\frac{n_j}{n}
\quad\Longrightarrow\quad
\hat f_j=\frac{n_j}{nh_j}
$$

把所有柱面积相加就得到 1。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_j=\frac{n_j}{n\,h_j}
$$

**符号解释：** $n_j$ 是第 $j$ 个箱中的样本数，$h_j$ 是箱宽；密度直方图总面积为 1。

**代码对应：** 固定 `bins` 或箱边界比较不同组；密度口径使用 `stat='density'` 或对应参数。

**使用边界：** 箱宽改变会显著改变形状；不同样本量的组不宜直接比较原始频数。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.histogram(orders, x="order_value", nbins=20, title="订单金额分布")
fig.update_layout(
    xaxis_title="客单价（元）", yaxis_title="订单数", bargap=0.04
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：上面这张直方图把订单金额 order_value 画成了 20 个分箱。试着只改一个分箱参数或换一个数据字段，观察分布形状的变化：把 nbins 从 20 改成 12 或 30，看看分箱变粗或变细后分布轮廓怎么变；也可以把 x 换成 items（购买件数），看离散数值的计数分布。下面留好了脚手架，把分箱数填进 bin_count、要画的字段填进 x_col，先自己跑一遍体会区别，再点开答案对照自检。


In [ ]:
try:
    # 请在下方填写代码：练一练，把 nbins 改成 12 或 30，或把 x 换成 items，再观察分布变化。

    import plotly.express as px
    import pandas as pd

    bin_count = 20  # TODO: 可改成 12 或 30
    x_col = "order_value"  # TODO: 可改成 "items"

    fig = px.histogram(
        orders, x=x_col, nbins=bin_count, title="直方图：分箱与字段对比"
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.histogram(
    orders,
    x="order_value",
    color="category",
    nbins=20,
    barmode="overlay",
    opacity=0.6,
    histnorm="probability density",
    marginal="box",
    title="品类客单价分布",
)
fig.update_layout(
    xaxis_title="客单价（元）", yaxis_title="概率密度", legend_title="品类"
)
fig.show()


## 参数说明

- nbins：分箱数
- histnorm：归一化
- barmode：overlay/stack
- marginal：边缘图


## 结果解读

Hover显示每个分箱范围和数量；改变分箱确认分布结论稳定。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 分组时分箱边界不一致
- 图例筛选后忘记样本量改变
- 过度依赖默认分箱


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：按渠道分色直方图，观察重叠分布
    # 【目标】用颜色分层同一变量的分布，看各渠道重叠程度。
    import plotly.express as px

    # 起点示例(已可运行)：加 color="channel"，用半透明色叠加。
    fig = px.histogram(
        orders,
        x="order_value",
        nbins=20,
        color="channel",
        opacity=0.7,
        title="分渠道订单金额分布",
    )
    fig.update_layout(
        xaxis_title="客单价（元）", yaxis_title="订单数", bargap=0.04
    )
    fig.show()

    # ---- 反思记录：分色直方图下，渠道分布是否重叠 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互直方图查看分布，并通过悬浮和图例比较分类组。


### 你已经掌握

- 判断交互直方图（px.histogram）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `nbins` | 分箱数 |
| `histnorm` | 归一化 |
| `barmode` | overlay/stack |
| `marginal` | 边缘图 |


### 需要注意

- 分组时分箱边界不一致
- 图例筛选后忘记样本量改变
- 过度依赖默认分箱


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import pandas as pd
import plotly.express as px

# 完整答案：把分箱数改为 12，并把 x 换成 items（购买件数），突出离散分布对比
bin_count = 12
x_col = "items"
fig = px.histogram(
    orders, x=x_col, nbins=bin_count, title="购买件数分布——分箱为 12"
)
fig.update_layout(xaxis_title="购买件数", yaxis_title="订单数", bargap=0.04)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
fig = px.histogram(
    orders,
    x="items",
    color="channel",
    barmode="group",
    category_orders={"items": sorted(orders["items"].unique())},
    title="渠道购买件数分布",
)
fig.update_layout(
    xaxis_title="购买件数", yaxis_title="订单数", legend_title="渠道"
)
fig.show()
